**<h1 align="center">SBERT Baseline Training</h1>**

This notebook trains and evaluates a Sentence-BERT (SBERT) bi-encoder for career transition prediction framed as CV–job matching. It establishes an ontology-free baseline using ESCO job descriptions and JobHop career data, serving as a reference for comparison with ontology-augmented models.

# Table of Contents
* [1. Imports & Setup](#chapter1)
* [2. Load Data](#chapter2)
* [3. Initial Analysis](#chapter3)
* [4. Training Pair Construction and Retrieval Evaluation](#chapter4)
* [5. Model Setup & Training](#chapter5)

<a class="anchor" id="chapter1"></a>

# 1. Imports & Setup

</a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import sys
import pandas as pd
import torch
from torch.utils.data import DataLoader

In [ ]:
BASE = "/content/drive/MyDrive/NOVA IMS/Year 2/Thesis"
if BASE not in sys.path:
    sys.path.insert(0, BASE)

from thesis_utility import (
    build_train_examples_with_meta,
    build_ir_eval_unique_jobs_df,
    train_sbert_with_dataloader,
)

In [ ]:
# Check if running on GPU
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch: 2.9.0+cu126
CUDA available: True
GPU: Tesla T4


In [ ]:
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
DATA_DIR = f"{BASE}/Data"
OUT_DIR = f"{BASE}/Models"

<a class="anchor" id="chapter2"></a>

# 2. Load Data

</a>

In [ ]:
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
val_df = pd.read_csv(f"{DATA_DIR}/val.csv")

<a class="anchor" id="chapter3"></a>

# 3. Initial Analysis

</a>

In [ ]:
train_df.head(2)

,person_id,t,cv_codes,cv_labels,cv_skills_lol,cv_text,target_code,target_label,preferredLabel,jd_text,occupationUri,iscoGroup
0,0,1,['2353.1'],['language school teacher'],"[['assessment processes', 'learning difficulti...",Work experience: language school teacher. Skil...,3512.1,ict help desk agent,ict help desk agent,ICT help desk agent. ICT help desk agents prov...,http://data.europa.eu/esco/occupation/aaeec9a7...,3512
1,0,2,"['2353.1', '3512.1']","['language school teacher', 'ict help desk age...","[['assessment processes', 'learning difficulti...",Work experience: language school teacher; ict ...,3512.2,ict help desk manager,ict help desk manager,ICT help desk manager. ICT help desk managers ...,http://data.europa.eu/esco/occupation/1242d99a...,3512


In [ ]:
print("Unique targets:", train_df["target_label"].nunique())
print("Total pairs:", len(train_df))

train_df["target_label"].value_counts().describe()

Unique targets: 2912
Total pairs: 707633


,count
count,2912.000000
mean,243.005838
std,1115.434370
min,1.000000
25%,13.000000
50%,41.000000
75%,141.250000
max,31548.000000


In [ ]:
train_df["target_label"].value_counts().head(10)

,count
target_label,
administrative assistant,31548
sales assistant,23027
shop assistant,18102
warehouse worker,16442
cashier,12617
kitchen assistant,12023
waiter/waitress,10964
building cleaner,10717
factory hand,9893


<a class="anchor" id="chapter4"></a>

# 4. Training Pair Construction and Retrieval Evaluation

</a>

This section constructs SBERT training pairs and defines the retrieval-based evaluation setup. Training examples consist of positive CV–job description pairs derived from observed transitions. Evaluation is framed as an information retrieval task, where each CV must retrieve the correct job from a deduplicated ESCO corpus, using standard ranking metrics.

In [ ]:
# Build training examples from train (learn model parameters from positive CV–JD pairs)
train_examples, _, _ = build_train_examples_with_meta(train_df)

In [ ]:
# Build evaluator on validation (evaluate retrieval performance during training)
ir_evaluator = build_ir_eval_unique_jobs_df(val_df, name="val-ir")

In [ ]:
print("Train examples:", len(train_examples))
print("Val queries:", len(ir_evaluator.queries))
print("Val corpus (unique jobs):", len(ir_evaluator.corpus))

Train examples: 539077
Val queries: 88489
Val corpus (unique jobs): 2501


The model is trained on over 539k observed career transitions and evaluated on 88k CV queries, each requiring retrieval of the correct occupation among approximately 2.5k possible jobs.

<a class="anchor" id="chapter5"></a>

# 5. Model Setup & Training

</a>

This section defines the training configuration for the SBERT baseline model. The model is initialized from all-MiniLM-L6-v2 and trained on the training split using MultipleNegativesRankingLoss, where each CV–job pair is treated as positive and other jobs in the batch act as implicit negatives.

Training is performed for a single epoch on a large dataset (≈700k pairs), which is sufficient given the use of a pre-trained encoder. A linear warm-up schedule is applied during early training to stabilize optimization, with performance monitored on the validation set.

The resulting model is a bi-encoder that maps CVs and job descriptions into a shared embedding space, enabling CV–job matching via cosine similarity. This serves as an ontology-free SBERT baseline for comparison with subsequent models.

In [ ]:
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=128, drop_last=True)

In [ ]:
baseline_path = train_sbert_with_dataloader(
    train_dataloader=train_dataloader,
    ir_evaluator=ir_evaluator,
    run_name="sbert_baseline_epoch1_bs128",
    out_dir=MODELS_DIR,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Val-ir Cosine Accuracy@1,Val-ir Cosine Accuracy@3,Val-ir Cosine Accuracy@5,Val-ir Cosine Accuracy@10,Val-ir Cosine Precision@1,Val-ir Cosine Precision@3,Val-ir Cosine Precision@5,Val-ir Cosine Precision@10,Val-ir Cosine Precision@20,Val-ir Cosine Recall@1,Val-ir Cosine Recall@3,Val-ir Cosine Recall@5,Val-ir Cosine Recall@10,Val-ir Cosine Recall@20,Val-ir Cosine Ndcg@10,Val-ir Cosine Mrr@10,Val-ir Cosine Map@10
2105,4.082900,No log,0.021166,0.050097,0.070856,0.107019,0.021166,0.016699,0.014171,0.010702,0.007725,0.021166,0.050097,0.070856,0.107019,0.154505,0.057794,0.042853,0.042853
4210,4.049800,No log,0.020861,0.050255,0.071636,0.108307,0.020861,0.016752,0.014327,0.010831,0.007837,0.020861,0.050255,0.071636,0.108307,0.156743,0.058274,0.043075,0.043075
4211,4.049800,No log,0.020861,0.050255,0.071636,0.108307,0.020861,0.016752,0.014327,0.010831,0.007837,0.020861,0.050255,0.071636,0.108307,0.156731,0.058274,0.043075,0.043075
